# Competencia 01 — Cuaderno de laboratorioEste notebook **no contiene el pipeline**: lo *ejecuta y lo muestra*.La lógica vive en `pipeline/*.py`. Acá se la corre, se inspeccionan las salidas y seexplora la base con SQL, al estilo de `z402_Feature_Engineering_en_SQL.ipynb`.## Por qué el pipeline está en `.py` y no en celdasTres razones, en orden de peso:1. **Regla 11 del proyecto.** Lo que tarda más de ~15 minutos no puede vivir en una   pestaña de Chrome. Un `.py` se lanza con `nohup python3 -u pipeline/p01_base.py &`,   escribe log a archivo y sobrevive a que se cierre la laptop, se caiga internet o   muera una VM spot. Una celda de notebook, no. Eso fue lo que hizo caer la cursada   anterior.2. **Los gates tienen que poder matar la corrida.** Un `raise AssertionError` en un   script detiene el proceso y el código de salida distinto de cero lo delata. En un   notebook, una celda que falla no impide ejecutar la siguiente, y el pipeline sigue   con datos malos.3. **Reproducibilidad para la cátedra.** Tienen que poder replicar la entrega desde el   crudo. `python3 pipeline/p01_base.py && python3 pipeline/p03_fabrica.py && ...` es   una orden; "abrí el notebook y corré las celdas en orden, salteando la 47" no lo es.## Qué sí va en notebook- Esto: correr las etapas y **ver** las salidas.- La exploración con SQL, que es interactiva por naturaleza.- El EDA y la investigación (ya están en `EDA_limpio.ipynb` y `competencia.ipynb`).- El material de estudio para el examen del 10 de diciembre.El reparto es: **el `.py` es el instrumento, el notebook es el cuaderno de laboratorio.**

## EntornoSe conecta al **mismo archivo DuckDB** que escribió el pipeline. No se recalcula nada:el notebook lee lo que las etapas dejaron en disco.

In [ ]:
import sysfrom pathlib import Pathimport pandas as pdimport duckdbPROYECTO = Path.cwd()sys.path.insert(0, str(PROYECTO / "pipeline"))import p00_config as cfgimport p02_fuga as fugapd.set_option("display.max_rows", 200)pd.set_option("display.float_format", lambda x: f"{x:,.4f}")print("base DuckDB:", cfg.BASE_DUCKDB)print("existe:", cfg.BASE_DUCKDB.is_file())

In [ ]:
# Magics de SQL, igual que en los scripts de la cátedra (z402).# La diferencia: en vez de duckdb:/// en memoria, se apunta al archivo que# dejó la etapa 01, así las consultas ven las tablas ya construidas.%load_ext sql%config SqlMagic.autopandas = True%config SqlMagic.feedback = False%config SqlMagic.displaycon = False%sql duckdb:///{{cfg.BASE_DUCKDB}}

## Etapa 00 — La configuraciónFuente única de verdad. Antes esto estaba repetido en cinco archivos y dos de ellosse contradecían: `competenciav2.ipynb` había quedado con folds de gap 1 mientras`modelo_referencia.py` usaba gap 2.

In [ ]:
print(f"ganancia por acierto : ${cfg.GANANCIA_BAJA2:,}")print(f"costo del estímulo   : ${cfg.COSTO_ESTIMULO:,}")print(f"umbral de indiferencia: {cfg.UMBRAL_TEORICO}  = 1/{1/cfg.UMBRAL_TEORICO:.0f}")print()print(f"meses del dataset    : {cfg.MESES_DATASET}")print(f"meses con target     : {cfg.MESES_TARGET_MADURO}")print(f"mes a predecir       : {cfg.MES_SCORING}")print()print(f"semillas de ENTREGA  : {cfg.SEMILLAS_ENTREGA}")print(f"semillas de MEDICIÓN : {len(cfg.SEMILLAS_MEDICION)} en total")print()for f in cfg.FOLDS:    cfg.verificar_gap(f)    gap = cfg.indice_mes(f["validacion"]) - cfg.indice_mes(max(f["train"]))    print(f"fold {f['nombre']:6s}: train {str(f['train']):22s} -> valida {f['validacion']}  (gap {gap})")

### Por qué existe `indice_mes()``foto_mes` es un entero con forma `AAAAMM`. Restarlo directamente funciona dentro delmismo año y **falla al cruzarlo**: diciembre y enero son meses contiguos pero la restaingenua da 89. Sin convertir a un índice continuo de meses, cualquier verificación degap se rompe silenciosamente en el cambio de año — que es exactamente lo que va a pasaren la Última Competencia, que abarca 18 meses.

In [ ]:
for a, b in [(202104, 202106), (202112, 202201), (202108, 202203)]:    print(f"{a} -> {b}:  distancia real {cfg.indice_mes(b)-cfg.indice_mes(a):3d}  |  "          f"resta ingenua {b-a:6d}")

## Etapa 01 — Carga y construcción del targetEl crudo **no trae la clase**. La baja de un cliente no es un campo: es el hecho de queel cliente deja de aparecer. La etiqueta del mes M se deduce mirando la presencia enM+1 y M+2:| presente en M+1 | presente en M+2 | clase | lectura ||---|---|---|---|| no | — | `BAJA+1` | ya se fue, llegamos tarde || sí | no | **`BAJA+2`** | **esto es lo que se predice** || sí | sí | `CONTINUA` | sigue siendo cliente |Para correr la etapa desde la terminal:```bashpython3 -u pipeline/p01_base.py```Acá solo se lee el resumen que dejó.

In [ ]:
resumen = pd.read_csv(cfg.SALIDAS / "p01_resumen_target.csv")resumen

Dos lecturas de esa tabla:- **`BAJA+2` es rarísimo: entre 0,531 % y 0,698 %.** El problema está fuertemente  desbalanceado y por eso la métrica no puede ser accuracy.- **202107 y 202108 están sin etiqueta, y es correcto que lo estén.** Julio permite  identificar `BAJA+1` (se ve quién desaparece en agosto) pero no separar `BAJA+2` de  `CONTINUA`, porque para eso haría falta septiembre. Agosto no tiene nada: es el mes a  predecir. Ninguno de los dos puede ser conjunto supervisado, pero **julio sí aporta  historia** para construir los lags de agosto.

### Verificación de granularidadEl par `(numero_de_cliente, foto_mes)` es la clave primaria compuesta. Si se repitiera,todo lo que viene después se rompe **en silencio**: `lag()` y `lead()` asumen una filapor cliente por mes, así que con duplicados `lead(presencia, 1)` no devuelve "el messiguiente" sino "la otra fila del mismo mes", y el target sale mal sin tirar ningúnerror. Cero filas es la respuesta que se busca.

In [ ]:
%%sqlselect numero_de_cliente, foto_mes, count(*) as nfrom basegroup by numero_de_cliente, foto_meshaving count(*) > 1

### El target, visto como pivotEl `PIVOT` de DuckDB es la forma más corta de ver la distribución de clases por mes.

In [ ]:
%%sqlpivot baseon clase_ternariausing count(numero_de_cliente)group by foto_mesorder by foto_mes

### Por qué el `CROSS JOIN` y no un `lead()` directoHay **23 clientes que desaparecen y vuelven**. Un `lead()` sobre las filas que existenleería mayo como "el mes siguiente de marzo" para un cliente que faltó en abril, y loetiquetaría `CONTINUA` cuando en realidad desapareció y eso lo hace `BAJA+1` en marzo.La solución es **materializar explícitamente la ausencia**: producto cartesianocliente × mes, marcar con `mes_0 = 1` dónde el cliente apareció de verdad, y reciénentonces aplicar `lead()`. Así la ventana camina el calendario completo y no la listade filas existentes.Es poco volumen —23 clientes— pero es la diferencia entre un target correcto y uno quemiente en los casos raros. Y en 18 meses los huecos van a ser muchos más.

In [ ]:
%%sql-- Un cliente con hueco: se ve el mes faltante y cómo quedó etiquetado el mes previo.with presencia as (    select numero_de_cliente,           count(*) as presente,           (max(foto_mes)//100)*12 + (max(foto_mes)%100)         - ((min(foto_mes)//100)*12 + (min(foto_mes)%100)) + 1 as abarcados    from base group by numero_de_cliente),con_hueco as (    select numero_de_cliente from presencia    where presente < abarcados limit 1)select b.numero_de_cliente, b.foto_mes, b.clase_ternaria, b.ctrx_quarterfrom base b join con_hueco h using (numero_de_cliente)order by b.foto_mes

## Etapa 02 — Los gates de fugaSeis fugas distintas, cada una atajada en otro lugar:| | Fuga | Dónde se ataja ||---|---|---|| **F1** | Ventanas de resultado solapadas (train a 1 mes de la validación) | `verificar_gap()` || **F2** | Etiqueta en el mes de scoring | GATE scoring en p01 || **F3** | Target propagado entre meses del mismo cliente | `gate_target_no_propagado()` || **F4** | `numero_de_cliente` / `foto_mes` como predictores | `gate_columnas_prohibidas()` || **F5** | Feature calculada con datos del futuro | `gate_sin_fuga_de_futuro()` || **F6** | Fuga de **selección** (elegir y medir en el mismo lugar) | protocolo, no código |

In [ ]:
con = duckdb.connect(str(cfg.BASE_DUCKDB), read_only=True)fuga.gate_target_no_propagado(con)fuga.gate_consistencia_baja2(con)fuga.gate_columnas_prohibidas(["ctrx_quarter", "mrentabilidad", "cpayroll_trx"])for f in cfg.FOLDS:    cfg.verificar_gap(f)print(f"[02] GATE F1 OK: los {len(cfg.FOLDS)} folds tienen gap >= {cfg.MESES_MADURACION}")

### La prueba de que el target no está propagadoSi alguien copiara la etiqueta de junio hacia todas las filas del cliente, cada clientetendría **una sola** clase distinta. Que haya clientes con dos y con tres demuestra quela clase pertenece al par `(cliente, mes)`.La trayectoria que describen las notas de clase está en los datos.

In [ ]:
%%sqlwith t as (    select numero_de_cliente,           string_agg(clase_ternaria, ' -> ' order by foto_mes) as trayectoria    from base where clase_ternaria is not null    group by numero_de_cliente)select trayectoria, count(*) as clientesfrom t group by trayectoriaorder by clientes desc limit 10

### Control de construcción concluyenteTodo `BAJA+2` en el mes M **tiene que ser** `BAJA+1` en M+1. Se deduce de la definición:`BAJA+2` en M significa presente en M+1 y ausente en M+2; visto desde M+1 eso esexactamente "ausente el mes que viene", o sea `BAJA+1`.Si aparece cualquier otra clase, el `lead()` está desalineado. Da 100 %.

In [ ]:
%%sqlwith pares as (    select b.clase_ternaria as clase_mes_siguiente    from base a    join base b on b.numero_de_cliente = a.numero_de_cliente               and (b.foto_mes//100)*12 + (b.foto_mes%100)                 = (a.foto_mes//100)*12 + (a.foto_mes%100) + 1    where a.clase_ternaria = 'BAJA+2')select clase_mes_siguiente, count(*) as casosfrom pares group by clase_mes_siguiente

### La estrategia de validación, y por qué **no** es k-foldEsto es una decisión de modelado, no de infraestructura: hay que poder defenderla.

In [ ]:
fuga.imprimir_estrategia_de_validacion()

## Experimento 01 — La regla de corte**Hipótesis:** el umbral teórico 0,025 no maximiza la ganancia, porque lasprobabilidades de LightGBM no están calibradas en la cola superior. Existe una regla decorte, con su parámetro elegido **fuera** del mes que se mide, que gana más.**Resultado: hipótesis rechazada.** Y el resultado negativo vale más que uno positivo,porque explica *por qué* no se puede tunear nada en esta ventana.

In [ ]:
reglas = pd.read_csv(PROYECTO / "experimentos/salidas/exp01_reglas_corte.csv")tabla = (reglas.pivot_table(index="regla", columns="medicion",                            values="ganancia", aggfunc="mean") / 1e6).round(2)base_ = tabla.loc["R0_umbral_teorico"]tabla["delta_junio"] = (tabla["junio"] - base_["junio"]).round(2)tabla["delta_mayo"]  = (tabla["mayo"]  - base_["mayo"]).round(2)tabla

**La causa:** el k óptimo es ~8.200 en mayo y ~14.100 en junio. Se mueve **+71 %**mientras la tasa de bajas se mueve +26 %, así que no es un efecto de prevalencia que sepueda normalizar. Llevar el k de junio a mayo sobre-estimula y cuesta 43 M.**La virtud de 0,025 no es ser óptimo: es no tener ningún parámetro que deba transferir.**La misma cosa pasa con la cantidad de iteraciones: mayo pide entrenar más largo, juniopide cortar en 150, **las dos con p < 0,01**. Mismo modelo, mismos parámetros, mesescontiguos.

In [ ]:
met = pd.read_csv(PROYECTO / "experimentos/salidas/exp01_metricas.csv")(met.pivot_table(index="iteraciones", columns="fold",                 values="ganancia_umbral_teorico", aggfunc="mean") / 1e6).round(2)

In [ ]:
# El ruido que domina la comparación entre dos juegos de features NO es el de la# muestra de evaluación (es el mismo conjunto de clientes en los dos brazos):# es el de la semilla del árbol.(met[met.iteraciones == 150]    .groupby("fold")["ganancia_umbral_teorico"]    .agg(media="mean", desvio="std", minimo="min", maximo="max", semillas="count")    .div(1e6).round(2))

## Qué sigue**Etapa 03 — la fábrica de features.** SQL generado programáticamente desde listas decampos (patrón `z402`, celdas 27-29): lags 1-3, deltas relativos, medias/mín/máxmóviles, `regr_slope`, rachas de caída y eventos de prendido/apagado.Con una exigencia de arquitectura que sale del gate **F5**: el SQL **no puede apuntar aun nombre de tabla fijo**, tiene que recibir la tabla de origen como parámetro. Así elgate puede reconstruir exactamente las mismas features sobre una tabla truncada en elmes M y comparar fila por fila. Si una feature solo usa meses ≤ M, truncar en M no puedecambiarla: cualquier diferencia es **prueba** de que leyó el futuro.Después, **etapa 04** (canaritos: cortar lo que no supera al ruido puro) y **etapa 05**(el banco de medición: 20 semillas × 2 folds, apareado, Wilcoxon).